# Plato Input Catalogue (PIC) 2.1.0.1 

This notebook explores the PIC 2.1.0.1 catalogue and its sub-PICs. Ultimately the goal of this work is to produce feather-format files that can used by `picsim` to generate on-demand stellar catalogues to be used by `platonium`. The output feather-format catalogues from this notebook are placed on the KU Leuven FTP located at `/STER/platoman/ftp`. From that location, `picsim` automatically downloads these files upon first execution and you can enjoy the user-friendly interface of `picsim` to generate your sub-PIC catalogues used by `PlatoSim`.

**Prerequsites** 

To run this notebook you need to download the PIC 2.1.0.1 and place the following files alonside this notebook:

- `LOPS2PICtarget2.1.0.1-t-fg-c-scv.fits`
- `LOPS2PICcontaminant2.1.0.1-t-fg-c-scv.fits`

**Documentation** 

The following technical notes helps you understand the PIC:

- `LOPS2PIC2.1.0.1_DataDefinitionDocument_PLATO-SSDC-PDC-DD-0003.pdf`
- `LOPS2PIC2.1.0.1_ReleaseNote_PLATO-SSDC-PDC-DN-0001.pdf`
- `PLATO-DLR-PL-RP-0001_i4.3.pdf`
- `PLATO-DLR-PL-TN-0113_issue1.0_PLATO_instrument_spectral_response.pdf`
- `PLATO-KUL-PMC-TN-0001_i1.0_Science_Calibration_Stars_Input_Catalogue_Requirements_Justification.pdf`
- `PLATO_SCI_UPD_TN_0022_1.2.pdf`

**Last checked** 

This notebook was last functional with the PlatoSim version:

- `PlatoSim 3.7.0-191-g42fc64d4`

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib notebook

In [ ]:
# Default packages
import os
import sys
import glob

# PlatoSim default
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# PLATOnium default
from pathlib import Path
from astropy.io import fits
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy import units as u

# PlatoSim libraries
import platosim.plot      as pt
import platosim.utilities as ut
from platosim.pic          import PIC2101
from platosim.matplotlibrc import setup_paper
setup_paper(warning=False)

from IPython.display import display, HTML
display(HTML("<style>.container {width:80% !important; }</style>"))

Relative path to catalogues (if you have placed the catalogue somewhere else on your system, simple alter the following path):

In [ ]:
# path = Path('LOPS2PIC2.1.0.1-t-fg-c-scv/Data')
path = Path(os.getenv('PLATO_PROJECT_HOME')) / 'inputfiles/data_picsim'

---
## LOPS2 PIC 2.1.0.1 targets
---

This catalogue is the main target table, composed by the merge of four different target lists:

- `tPIC`: Core science stars (P1, P2, P4, P5) and stars with confirmed/candidate planet(s)
- `fgPIC`: Fine Guidance System (FGS) stars used to guarantee the attitude of the spacecraft
- `cPIC`: Instrument calibration stars
- `scvPIC`: Calibration and validation stars for both stellar and exoplanets science

Load target catalogue into data frame:

In [ ]:
filename_targets = path / 'LOPS2PICtarget2.1.0.1-t-fg-c-scv.fits'
filename_nsr     = path / 'LOPS2PICtargetNSR2.1.0.1-t-fg-c-scv.fits'

We will use the PIC utilities to fetch a few columns:

In [ ]:
pic = PIC2101(filename_targets, filename_nsr)

For the remaining columns we fetch them manually from a data frame:

In [ ]:
filename_targets = path / 'LOPS2PICtarget2.1.0.1-t-fg-c-scv.fits'
table = Table.read(filename_targets, format='fits')
df = table.to_pandas()
df.head()

Check the column names:

In [ ]:
df.columns

Define a new dataframe and copy over the columns we need:

In [ ]:
# Add Galactic coordinates
gal = SkyCoord(df.RAdeg, df.DEdeg, frame='icrs', unit=u.deg).galactic
# Create data frame
dt = pd.DataFrame()
dt['PIC']     = df.PICid
dt['gaiaDR3'] = pic.get_GDR3idnumber()
dt['l']       = gal.l.deg
dt['b']       = gal.b.deg
dt['ra']      = df.RAdeg
dt['dec']     = df.DEdeg
dt['pmra']    = df.pmRA
dt['pmdec']   = df.pmDE
dt['Pmag']    = df.PlatoMagNCAM
dt['PBmag']   = df.PlatoMagFCAMb
dt['PRmag']   = df.PlatoMagFCAMr
dt['M']       = df.Mass
dt['R']       = df.Radius
dt['Teff']    = df.Teff
dt['logg']    = pic.get_logg()
dt['ncams']   = df.BOLnCameraObsNCAM_T
dt['case']    = df.caseFlag
dt['source']  = df.PICmainSourceFlagBOL
dt['tPIC']    = df.tPICsourceFlagNCAM_BOL
dt['fgPIC']   = df.fgPICsourceFlag
dt['cPIC']    = df.cPICsourceFlag
dt['scvPIC']  = df.scvPICsourceFlag
dt.ncams.unique()

Remove stars ouside of visibility flower and save `picsim` target catalogue:

In [ ]:
dt0 = dt[dt.ncams != 0]
dt0 = dt0.reset_index(drop=True)
dt0.to_feather('PIC210_LOPS2_targets.ftr')

---
## LOPS2 PIC 2.1.0.1 contaminants
---

Open fits file and inspect it:

In [ ]:
filename = path / 'LOPS2PICcontaminant2.1.0.1-t-fg-c-scv.fits'
hdul = fits.open(filename, memmap=True)

Note that this catalogue contains 105,865,176 sources:

In [ ]:
hdul.info()

Check tabel column names

In [ ]:
hdul[1].header

Workaround to load huge data table into a data frame:

In [ ]:
dc = pd.DataFrame()
campos = {
    'PIC'     : 'PICcontaminantId',
    'gaiaDR3' : 'StarName',
    'ra'      : 'RAdeg', 
    'dec'     : 'DEdeg',
#     'pmra'    : 'pmRA',
#     'pmdec'   : 'pmDE',
    'Pmag'    : 'PlatoMagNCAM',
    'PBmag'   : 'PlatoMagFCAMb',
    'PRmag'   : 'PlatoMagFCAMr'
}
print('FECTHING COLUMNS FROM CONTAMINANT TABLE:')
print('-'*40)
for campo_dc, campo_pic in campos.items():
    # Print each column to follow progress
    print(campo_pic)
    col = hdul[1].data[campo_pic]
    # Fetch magnitude column and get bool indices for P < 19
    if campo_dc == 'Pmag':
        dx = pd.DataFrame({'Pmag': col})
        dex = dx.Pmag < 19
    # Fetch each column
    dc[campo_dc] = col.byteswap().view(col.dtype.newbyteorder())
print('-'*40)

Show the number of sources for different magnitude limits:

In [ ]:
dc19 = dc.loc[dex]
dc19.shape[0]/1e6

In [ ]:
dc17 = dc[dc.Pmag < 17]
dc17.shape[0]/1e6

In [ ]:
dc15 = dc17[dc17.Pmag < 15]
dc15.shape[0]/1e6

We make a cut to keep the catalogue managable and remove NaNs:

In [ ]:
dc17 = dc17.dropna(subset=['PIC', 'ra', 'dec', 'Pmag', 'PBmag', 'PRmag'])
dc17.pmra = dc17.pmra.replace(np.nan, 0)
dc17.pmdec = dc17.pmdec.replace(np.nan, 0)

Since the PIC 2.1.0.1 contaminant table do not include unique PIC identifiers related to each target star we need to generate this ourselves:

In [ ]:
dc_cut = ut.getContaminants(dt, dc17, column='PIC', radius=60)

In [ ]:
dc_cut = dc_cut.reset_index(drop=True)
dc_cut

Save `picsim` contaminant catalogue:

In [ ]:
dc_cut.to_feather('PIC210_LOPS2_contaminants.ftr')

---
## Plot the PIC target catalogue
---

All PIC targets:

In [ ]:
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Settings
    figsize=(9,9), title=f'{dt.shape[0]} PIC targets'
)

---
### Plots for tPIC

Plot `tPIC` targets:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['source'] & 4 == 4]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dx,
    # Settings
    figsize=(9,9), title=f'{dx.shape[0]} tPIC targets'
)

Plot all `P1` targets and their planets (candidates):

In [ ]:
dx = dt[(dt.tPIC == 1) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=20, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P1 targets'
)

In [ ]:
dx = dt[(dt.tPIC == 17) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P1 planets'
)

Plot all `P2` targets and their planets (candidates):

In [ ]:
dx = dt[(dt.tPIC == 3) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P2 targets'
)

In [ ]:
dx = dt[(dt.tPIC == 19) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P2 planets'
)

Plot all `P4` targets and their planets (candidates):

In [ ]:
dx = dt[(dt.tPIC == 8) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=20, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P4 targets'
)

In [ ]:
dx = dt[(dt.tPIC == 24) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P4 planets'
)

Plot all `P5` targets and their planets (candidates):

In [ ]:
dx = dt[(dt.tPIC == 4) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=1.5, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P5 targets'
)

In [ ]:
dx = dt[(dt.tPIC == 20) & (dt.ncams != 0)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} tPIC P5 planets'
)

---
### Plots for fgPIC

Plot `fgPIC` targets:

In [ ]:
dx = dt.loc[dt.source & 8 == 8]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} fgPIC targets'
)

Plot fgPIC `F-CAMb` targets:

In [ ]:
dx = dt.loc[dt.fgPIC & 1 == 1]

fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} fgPIC F-CAM blue targets'
)

Plot fgPIC `F-CAMr` targets:

In [ ]:
dx = dt.loc[dt.fgPIC & 2 == 2]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} fgPIC F-CAM red targets'
)

---
### Plots for cPIC

Plot `cPIC` targets:

In [ ]:
dx = dt
# dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['source'] & 16 == 16]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=34, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=5, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC targets'
)

Plot cPIC `F-CAM R1` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 1 == 1]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC F-CAM R1 targets'
)

Plot cPIC `F-CAM R2` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 2 == 2]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=20, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC F-CAM R2 targets'
)

Plot cPIC `F-CAM R3` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 4 == 4]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC F-CAM R3 targets'
)

Plot `cPIC F-CAM R4` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 8 == 8]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC F-CAM R4 targets'
)

Plot cPIC `F-CAM R5` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 16 == 16]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC F-CAM R5 targets'
)

Plot cPIC `N-CAM R1` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 32 == 32]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC N-CAM R1 targets'
)

Plot cPIC `N-CAM R2` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 64 == 64]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=10, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC N-CAM R2 targets'
)

Plot cPIC `N-CAM R3` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 128 == 128]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC N-CAM R3 targets'
)

Plot cPIC `N-CAM R4` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 256 == 256]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC N-CAM R4 targets'
)

Plot cPIC `N-CAM R5` sample:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['cPIC'] & 512 == 512]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} cPIC N-CAM R5 targets'
)

---
### Plots for scvPIC

Plot `scvPIC` targets:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['source'] & 32 == 32]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=10, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} scvPIC targets'
)
fig.savefig('skymap_LOPS2_scvPIC2101.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV1a` sample (eclipsing binaries):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 1 == 1]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1a targets (eclipsing binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV1a.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV1b` sample (astrometric binaries):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 2 == 2]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1b targets (astrometric binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV1b.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV1c` sample (wide binaries):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 4 == 4]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1c targets (wide binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV1c.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV1d` sample (HW Vir-type binaries):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 8 == 8]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1d targets (HW Vir-type binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV1d.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV1e` sample (wide WD binaries):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 16 == 16]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV1e targets (wide WD binaries)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV1e.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV2a` sample (legacy and benchmark):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 32 == 32]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV2a targets (legacy and benchmark)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV2a.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV2b` sample (legacy and benchmark):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 64 == 64]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=50, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV2b targets (legacy and benchmark)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV2b.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV3a` sample (photometric stable):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 128 == 128]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV3a targets (photometric stable)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV3a.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV3b` sample (photometric stable):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 256 == 256]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV3b targets (photometric stable)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV3b.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV4a` sample (solar-like pulsators):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 512 == 512]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=70, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV4a targets (solar-like pulsators)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV4a.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV4b` sample (solar-like pulsators):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 1024 == 1024]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=10, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV4b targets (solar-like pulsators)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV4b.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV5` sample (gamma Dor pulsators):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 2048 == 2048]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=30, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV5 targets (gamma Dor pulsators)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV5.png', bbox_inches='tight', dpi=200)

Plot scvPIC `SCV6` sample (transiting exoplanets):

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['scvPIC'] & 4096 == 4096]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=100, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} SCV6 targets (transiting exoplanets)'
)
fig.savefig('skymap_LOPS2_scvPIC2101_SCV6.png', bbox_inches='tight', dpi=200)

Lastly, we show all targets without (red) giants. The reason being that we have found that this catalogue is faulty and thus it I (Nicholas) reported it to the scvPIC working group. It should be solved for the next release of PIC 2.2.0.1. Here is the issue:

In [ ]:
dx = dt[(dt.ncams != 0)]
dx = dx.loc[dx['source'] & 32 == 32]
dx = dx[(dx.case != 4) & (dx.case != 6) & (dx.case != 10) & 
        (dx.case != 12) & (dx.case != 16) & (dx.case != 18)]
fig, ax = pt.plotPlatoFOV(
    # Plot skymap
    'LOPS2', system='galactic', fovSize=30, ncamStars=dt,
    # Plot stars
    raStars=dx.ra, decStars=dx.dec, c=dx.Pmag, s=20, 
    cmap='inferno', clabel=r'$\mathcal{P}$ [mag]', 
    # Settings
    figsize=(10,10), title=f'{dx.shape[0]} scvPIC targets (no giants)'
)